In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# Fix 1 — `cal-disp validate` compares the whole product

**Defect.** `src/cal_disp/validate.py` (gamma-release, `_compare_group` / `_compare_array`,
lines 163–276) compared only the filename fields, dimension names, shapes, the NaN pattern
and the data values of the `main`/`auxiliary` groups. It never looked at attributes
(`units`, `long_name`, `grid_mapping`, `_FillValue`), the CRS WKT, dtypes, the
`identification`/`metadata` values or the browse PNG; a missing group was swallowed
(`except (KeyError, OSError): pass`, line 101); `tolerance` was used as both `rtol` and
`atol` (line 264); pixels where the reference is exactly 0 were skipped (line 260), which
also skipped the millions of clipped `calibration_std` zeros; and the CLI default (1e-6)
differed from the function default (1e-4).

**Fix.** `compare_cal_products` now compares, group by group, the presence of every group,
the variable set, on-disk dtype and shape, all attributes (with a small documented
ignore-list: build versions, `_NCProperties`, `processing_start_datetime`, the embedded
`pge_runconfig`), the CRS as a `rasterio.crs.CRS`, the grid transform, all values (strings
and integers exactly, floats with separate `--rtol`/`--atol`, reference zeros included) and
the browse PNG (≤ 2048 px per side; only a warning when absent). One `DEFAULT_TOLERANCE`
(1e-6) is shared by the function, the CLI and the scripts. Differences are reported grouped
and the CLI exits 1 on any of them.

**Demonstrated here.** The golden product is copied to scratch and mutated one field at a
time with `h5py`; each mutation is run through the *old* comparison (checked out from
`gamma-release` commit `d0901b3` with `git show`) and the *new* one. The table at the end shows which
mutations the old validator missed. Every mutation is derived from the value the golden
currently holds (read it, then change it), so the notebook stays valid when the golden is
regenerated.

In [2]:
import importlib.util
import logging
import shutil
import subprocess
import time

import h5py
import numpy as np
import pandas as pd

from cal_disp.validate import compare_cal_products

golden_nc = sorted((GOLDEN / "golden_output").glob("OPERA_L4_DISP-CAL-S1_*.nc"))[0]
work_dir = SCRATCH / "08_validation_depth"
work_dir.mkdir(exist_ok=True)
test_nc = work_dir / golden_nc.name

# One working copy (~0.6 GB); mutations below are reverted in place
shutil.copy(golden_nc, test_nc)  # always a fresh copy
test_nc.with_suffix(".png").unlink(missing_ok=True)  # no browse image: only a warning
print(golden_nc.name, f"{golden_nc.stat().st_size / 1e6:.0f} MB")

OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T213458Z.nc 209 MB


## The pre-fix comparison, straight from `gamma-release`

`git show d0901b3:src/cal_disp/validate.py` (`gamma-release` before this fix) is written to scratch and imported as
`validate_old`, so both versions run on exactly the same files.

In [3]:
old_src = subprocess.run(
    ["git", "show", "d0901b3:src/cal_disp/validate.py"],
    cwd=REPO, check=True, capture_output=True, text=True,
).stdout
old_path = work_dir / "validate_old.py"
old_path.write_text(old_src)
spec = importlib.util.spec_from_file_location("validate_old", old_path)
validate_old = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate_old)

# Both validators log to the same logger; keep only the new one's failure lines
messages: list[str] = []


class _Collect(logging.Handler):
    def emit(self, record):
        if record.levelno >= logging.ERROR:
            messages.append(record.getMessage())


logging.getLogger("cal_disp.validate").addHandler(_Collect())
logging.getLogger("cal_disp.validate").setLevel(logging.ERROR)
logging.getLogger("validate_old").setLevel(logging.ERROR)


def run_both(label: str) -> dict:
    t0 = time.time()
    old_ok = validate_old.compare_cal_products(golden_nc, test_nc, tolerance=1e-6)
    t1 = time.time()
    messages.clear()
    new_ok = compare_cal_products(golden_nc, test_nc, rtol=1e-6, atol=1e-6)
    t2 = time.time()
    # The working copy has no browse PNG next to it (a warning only); the browse
    # check is shown separately below
    findings = [m for m in messages if m.startswith("  ") and "browse" not in m and ".png" not in m]
    return {
        "mutation": label,
        "old validate": "PASS" if old_ok else "FAIL",
        "new validate": "PASS" if new_ok else "FAIL",
        "new report (first line)": findings[0].strip() if findings else "",
        "old s": round(t1 - t0, 1),
        "new s": round(t2 - t1, 1),
    }

/u/aurora-r0/govorcin/01_OPERA/VLM/DISP_CAL/gamma_release/cal-disp/.claude/worktrees/agent-afc2178dd2aded541/.pixi/envs/dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Mutations

The current values are read from the working copy first; each mutation changes one of them
to something different (the neighbouring UTM zone, another unit string, `track_number + 1`,
the other float width, one pixel + 1e-5). Each is applied with `h5py`, both validators are
run, and the mutation is reverted. Dropping the `auxiliary` group cannot be reverted, so it
goes last.

In [4]:
from rasterio.crs import CRS

with h5py.File(test_nc) as f:
    orig_wkt = f["spatial_ref"].attrs["crs_wkt"]
    orig_units = f["calibration"].attrs["units"]
    orig_track = int(f["identification/track_number"][()])
    xs = f["identification/source_data_x_spacing"]
    xs_attrs, xs_value, xs_dtype = dict(xs.attrs), xs[()], xs.dtype
    # A finite pixel to perturb: first one found scanning rows from the middle
    ny = f["calibration"].shape[1]
    for iy in [*range(ny // 2, ny), *range(ny // 2)]:
        row = f["calibration"][0, iy, :]
        if np.isfinite(row).any():
            ix = int(np.argmax(np.isfinite(row)))
            orig_pixel = row[ix]
            break
    else:
        raise RuntimeError("the golden calibration layer has no finite pixel")

# Each new value is derived from the current one, so it always differs
orig_crs = CRS.from_wkt(orig_wkt)
orig_epsg = orig_crs.to_epsg()
if orig_epsg and 32601 <= orig_epsg <= 32760:  # UTM: the neighbouring zone
    zone = orig_epsg % 100
    new_epsg = orig_epsg + 1 if zone < 60 else orig_epsg - 1
else:
    new_epsg = 4326 if orig_epsg != 4326 else 32615
new_wkt = CRS.from_epsg(new_epsg).to_wkt()
new_units = "millimeters" if orig_units != "millimeters" else "meters"
new_track = orig_track + 1
new_dtype = np.dtype(np.float32 if xs_dtype == np.float64 else np.float64)
assert CRS.from_wkt(new_wkt) != orig_crs and new_units != orig_units and new_dtype != xs_dtype

print(f"golden now holds: CRS EPSG:{orig_epsg} | calibration units {orig_units!r} | track_number {orig_track}"
      f" | source_data_x_spacing {xs_dtype} | finite pixel {(iy, ix)} = {orig_pixel!r}\n")


def set_scalar(f, name, value, dtype, attrs):
    """Recreate a scalar dataset with another dtype (h5py cannot cast in place)."""
    del f[name]
    ds = f.create_dataset(name, data=np.array(value, dtype=dtype))
    for k, v in attrs.items():
        ds.attrs[k] = v


mutations = [
    (
        f"CRS WKT: EPSG:{orig_epsg} -> EPSG:{new_epsg}",
        lambda f: f["spatial_ref"].attrs.__setitem__("crs_wkt", new_wkt),
        lambda f: f["spatial_ref"].attrs.__setitem__("crs_wkt", orig_wkt),
    ),
    (
        f"calibration units: {orig_units} -> {new_units}",
        lambda f: f["calibration"].attrs.__setitem__("units", new_units),
        lambda f: f["calibration"].attrs.__setitem__("units", orig_units),
    ),
    (
        f"identification/track_number: {orig_track} -> {new_track}",
        lambda f: f["identification/track_number"].__setitem__((), new_track),
        lambda f: f["identification/track_number"].__setitem__((), orig_track),
    ),
    (
        f"calibration[{iy}, {ix}] += 1e-5",
        lambda f: f["calibration"].__setitem__((0, iy, ix), orig_pixel + np.float32(1e-5)),
        lambda f: f["calibration"].__setitem__((0, iy, ix), orig_pixel),
    ),
    (
        f"identification/source_data_x_spacing dtype: {xs_dtype} -> {new_dtype}",
        lambda f: set_scalar(f, "identification/source_data_x_spacing", xs_value, new_dtype, xs_attrs),
        lambda f: set_scalar(f, "identification/source_data_x_spacing", xs_value, xs_dtype, xs_attrs),
    ),
    (
        "drop the auxiliary group",
        lambda f: f.__delitem__("auxiliary"),
        None,
    ),
]

rows = [run_both("no mutation (identical copy)")]
for label, apply, revert in mutations:
    with h5py.File(test_nc, "a") as f:
        apply(f)
    rows.append(run_both(label))
    if revert is not None:
        with h5py.File(test_nc, "a") as f:
            revert(f)
    print(f"{rows[-1]['mutation']:66s} old={rows[-1]['old validate']}  new={rows[-1]['new validate']}")

golden now holds: CRS EPSG:32615 | calibration units 'meters' | track_number 34 | source_data_x_spacing float64 | finite pixel (3866, 482) = np.float32(0.11412716)



CRS WKT: EPSG:32615 -> EPSG:32616                                  old=PASS  new=FAIL


calibration units: meters -> millimeters                           old=PASS  new=FAIL


identification/track_number: 34 -> 35                              old=PASS  new=FAIL


Data mismatch for 'calibration':


...

  Tolerance: 1e-06


calibration[3866, 482] += 1e-5                                     old=FAIL  new=FAIL


identification/source_data_x_spacing dtype: float64 -> float32     old=PASS  new=FAIL


auxiliary group mismatch: reference=True, test=False


drop the auxiliary group                                           old=FAIL  new=FAIL


## Result: mutation × {old, new}

In [5]:
pd.set_option("display.max_colwidth", 120)
table = pd.DataFrame(rows).set_index("mutation")
table

,old validate,new validate,new report (first line),old s,new s
mutation,,,,,
no mutation (identical copy),PASS,PASS,,6.3,5.2
CRS WKT: EPSG:32615 -> EPSG:32616,PASS,FAIL,"spatial_ref: CRS EPSG:32616, expected EPSG:32615",5.2,5.1
calibration units: meters -> millimeters,PASS,FAIL,"calibration attribute 'units': 'millimeters', expected 'meters'",5.2,5.1
identification/track_number: 34 -> 35,PASS,FAIL,"track_number: 35, expected 34",5.2,5.1
"calibration[3866, 482] += 1e-5",FAIL,FAIL,"calibration: 1 of 48396065 values beyond tolerance (max |diff|=1e-05, mean |diff|=2.07e-13, rtol=1e-06, atol=1e-06);...",5.2,6.1
identification/source_data_x_spacing dtype: float64 -> float32,PASS,FAIL,"source_data_x_spacing: dtype float32, expected float64",5.2,5.2
drop the auxiliary group,FAIL,FAIL,group missing from the test product,5.1,5.1


In [6]:
missed = table[(table["old validate"] == "PASS") & (table["new validate"] == "FAIL")]
print(f"{len(missed)} of {len(mutations)} mutations passed the old validator and are caught now:")
for name in missed.index:
    print("  -", name)
assert table.loc["no mutation (identical copy)", "new validate"] == "PASS"
assert (table.drop("no mutation (identical copy)")["new validate"] == "FAIL").all()

4 of 6 mutations passed the old validator and are caught now:
  - CRS WKT: EPSG:32615 -> EPSG:32616
  - calibration units: meters -> millimeters
  - identification/track_number: 34 -> 35
  - identification/source_data_x_spacing dtype: float64 -> float32


## Browse image

The new validator also checks the PNG next to the test product: it must be at most 2048 px on
each side (a missing PNG is only a warning). The old validator never looked at it. Shown
with the golden product's own PNG and with an oversized PNG placed next to the working copy
(compared with `group="main"`, since its `auxiliary` group was dropped above).

In [7]:
from PIL import Image

from cal_disp.validate import BROWSE_MAX_DIM


def browse_check(reference, test):
    messages.clear()
    ok = compare_cal_products(reference, test, group="main")
    png = test.with_suffix(".png")
    with Image.open(png) as img:
        size = img.size
    print(f"browse {size[0]}x{size[1]} px -> new validate: {'PASS' if ok else 'FAIL'}")
    for m in messages:
        if ".png" in m:
            print("   ", m.strip())


# 1. The golden product with its own browse image
browse_check(golden_nc, golden_nc)

# 2. An oversized browse image next to the (otherwise identical) working copy
Image.new("RGB", (BROWSE_MAX_DIM + 1, 64)).save(test_nc.with_suffix(".png"))
browse_check(golden_nc, test_nc)
old_ok = validate_old.compare_cal_products(golden_nc, test_nc, tolerance=1e-6, group="main")
print("old validate with the oversized browse image:", "PASS" if old_ok else "FAIL")

browse 2048x1673 px -> new validate: PASS


browse 2049x64 px -> new validate: FAIL
    OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T213458Z.png is 2049x64 px; each side must be <= 2048 px


old validate with the oversized browse image: PASS
